# Sampling Strategies

This notebook demonstrates all 5 built-in sampling strategies for cost-effective production evaluation.

**What you'll learn:**

1. **Why Sample** — Cost math and when 100% evaluation makes sense
2. **Random Sampling** — Unbiased general monitoring
3. **Stratified Sampling** — Fairness across user segments
4. **Score-Based Sampling** — Focus budget on failures
5. **Recent-First Sampling** — Post-deployment regression detection
6. **No Sampling** — Full audit for small datasets or incidents
7. **Comparison Table** — When to use which strategy
8. **Scheduling Patterns** — Hourly, nightly, and post-incident
9. **GitHub Actions Automation** — Cron-based evaluation workflows

Every cell is **runnable**. Make sure you've completed the [Overview notebook](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/overview.ipynb) first.

---

## Setup

In [ ]:
!pip install -q fasteval-core fasteval-langfuse

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"] = userdata.get("LANGFUSE_HOST")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Keys loaded from Colab Secrets.")
except (ImportError, Exception):
    # os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
    # os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
    # os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
    # os.environ["OPENAI_API_KEY"] = "sk-..."
    print("Using environment variables.")

In [ ]:
import fasteval as fe
from fasteval_langfuse import langfuse_traces, configure_langfuse, LangfuseConfig
from fasteval_langfuse.sampling import (
    NoSamplingStrategy,
    RandomSamplingStrategy,
    StratifiedSamplingStrategy,
    ScoreBasedSamplingStrategy,
    RecentFirstSamplingStrategy,
)

configure_langfuse(LangfuseConfig(auto_push_scores=True))
print("All 5 strategies imported and ready.")

---

## 1. Why Sample?

Every metric evaluation = 1 LLM judge call. The costs add up fast:

| Daily Traces | Metrics | Judge Calls/Day | GPT-4o-mini Cost* | GPT-4o Cost* |
|-------------|---------|-----------------|-------------------|--------------|
| 1,000 | 1 | 1,000 | ~$0.30 | ~$5.00 |
| 1,000 | 3 | 3,000 | ~$0.90 | ~$15.00 |
| 10,000 | 3 | 30,000 | ~$9.00 | ~$150.00 |
| 10,000 | 5 | 50,000 | ~$15.00 | ~$250.00 |

*\*Approximate. Actual cost depends on trace length and metric complexity.*

**Sampling at 2%** of 10k traces with 3 metrics = 600 calls instead of 30,000. That's a **98% cost reduction** while still getting statistically meaningful quality signals.

### When NOT to sample

- **Small projects**: < 100 traces/day — just evaluate everything
- **Post-incident audits**: You need to see every trace in the incident window
- **Dataset evaluations**: Datasets are already curated; evaluate all items

---

## 2. Random Sampling

**Best for:** General monitoring — get an unbiased view of overall quality.

Randomly selects `sample_size` traces from the fetched set. Use `seed` for reproducibility.

| Parameter | Type | Description |
|-----------|------|-------------|
| `sample_size` | `int` | Number of traces to evaluate |
| `seed` | `int \| None` | Random seed for reproducibility |

In [ ]:
@fe.correctness(threshold=0.8)
@langfuse_traces(
    time_range="last_24h",
    sampling=RandomSamplingStrategy(
        sample_size=5,
        seed=42,  # Same seed = same traces every run (for debugging)
    ),
)
def test_random_monitoring(trace_id, input, output, metadata):
    """Unbiased quality check of 5 random traces."""
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("Random Sampling (5 traces, seed=42):\n")
test_random_monitoring()

In [ ]:
# Without seed — different traces each run
@fe.correctness(threshold=0.8)
@langfuse_traces(
    time_range="last_24h",
    sampling=RandomSamplingStrategy(sample_size=5),  # No seed
)
def test_random_no_seed(trace_id, input, output, metadata):
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("Random Sampling (no seed — different each run):\n")
test_random_no_seed()

---

## 3. Stratified Sampling

**Best for:** Ensuring fairness across user segments — e.g., free vs. paid users, different regions, different query types.

Groups traces by a metadata key and samples evenly from each group. This prevents the most active segment from dominating the evaluation.

| Parameter | Type | Description |
|-----------|------|-------------|
| `strata_key` | `str` | Metadata key to group by (dot notation: `metadata.user_type`) |
| `samples_per_stratum` | `int` | Traces to sample from each group |

> **Colab Note:** If a group has fewer traces than `samples_per_stratum`, all traces from that group are included.

In [ ]:
@fe.correctness(threshold=0.8)
@fe.relevance(threshold=0.75)
@langfuse_traces(
    time_range="last_7d",
    sampling=StratifiedSamplingStrategy(
        strata_key="metadata.user_type",  # Group by: "free", "pro", "enterprise"
        samples_per_stratum=3,             # 3 traces per user type
    ),
)
def test_quality_by_user_type(trace_id, input, output, metadata):
    """Ensure quality is consistent across all user types."""
    user_type = metadata.get("user_type", "unknown")
    print(f"  [{user_type:>12}] {trace_id[:12]}: {str(input)[:50]}")
    fe.score(output, input=input)


print("Stratified Sampling (by user_type, 3 per group):\n")
test_quality_by_user_type()

In [ ]:
# Stratify by query category
@fe.correctness(threshold=0.8)
@langfuse_traces(
    filter_tags=["customer-support"],
    time_range="last_7d",
    sampling=StratifiedSamplingStrategy(
        strata_key="metadata.category",  # "billing", "technical", "account"
        samples_per_stratum=5,
    ),
)
def test_quality_by_category(trace_id, input, output, metadata):
    """Ensure support quality is consistent across query categories."""
    category = metadata.get("category", "unknown")
    print(f"  [{category:>12}] {trace_id[:12]}")
    fe.score(output, input=input)


print("Stratified Sampling (by category):\n")
test_quality_by_category()

---

## 4. Score-Based Sampling

**Best for:** Failure analysis — spend your evaluation budget where it matters most.

Oversamples traces with low existing scores (e.g., user ratings) and undersamples high-scoring traces. This is the most cost-effective strategy for finding and understanding failures.

| Parameter | Type | Description |
|-----------|------|-------------|
| `score_name` | `str` | Langfuse score to use for sampling (e.g., `user_rating`) |
| `low_score_threshold` | `float` | Scores below this are "low" |
| `low_score_rate` | `float` | Fraction of low-score traces to include (0.0–1.0) |
| `high_score_rate` | `float` | Fraction of high-score traces to include (0.0–1.0) |

> **Colab Note:** This requires traces in Langfuse to already have the score you're filtering on (e.g., `user_rating` from thumbs up/down feedback). If no scores exist, all traces are treated as unscored.

In [ ]:
# Investigate ALL failures, sample only 5% of successes
@fe.correctness(threshold=0.7)
@fe.relevance(threshold=0.7)
@langfuse_traces(
    time_range="last_7d",
    sampling=ScoreBasedSamplingStrategy(
        score_name="user_rating",
        low_score_threshold=3.0,  # Ratings 1-2 are "low"
        low_score_rate=1.0,       # Evaluate 100% of low-rated traces
        high_score_rate=0.05,     # Evaluate only 5% of high-rated traces
    ),
)
def test_failure_analysis(trace_id, input, output, context, metadata):
    """Focus evaluation budget on traces with poor user ratings."""
    if not context:
        context = metadata.get("docs") or metadata.get("retrieval_context")

    expected = metadata.get("expected_answer")
    print(f"  [{trace_id[:12]}] rating={metadata.get('user_rating', '?')}")
    fe.score(output, expected, context=context, input=input)


print("Score-Based Sampling (100% failures, 5% successes):\n")
test_failure_analysis()

In [ ]:
# Only evaluate failures (zero good traces)
@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="last_7d",
    sampling=ScoreBasedSamplingStrategy(
        score_name="user_rating",
        low_score_threshold=3.0,
        low_score_rate=1.0,
        high_score_rate=0.0,  # Skip ALL good traces
    ),
)
def test_only_failures(trace_id, input, output, metadata):
    """Evaluate ONLY traces with bad user ratings."""
    print(f"  [{trace_id[:12]}] rating={metadata.get('user_rating', '?')}: {str(input)[:50]}")
    fe.score(output, input=input)


print("Score-Based Sampling (failures only):\n")
test_only_failures()

---

## 5. Recent-First Sampling

**Best for:** Post-deployment monitoring — catch regressions as fast as possible by weighting recent traces more heavily.

Uses exponential decay so that newer traces are much more likely to be selected than older ones.

| Parameter | Type | Description |
|-----------|------|-------------|
| `decay_factor` | `float` | Decay rate (0.0–1.0). Higher = more recent bias. 0.8 is a good default. |

In [ ]:
@fe.correctness(threshold=0.8)
@langfuse_traces(
    time_range="last_24h",
    sampling=RecentFirstSamplingStrategy(
        decay_factor=0.8,  # Strong recent bias
    ),
    limit=20,  # Fetch 20, but recent ones are more likely to be selected
)
def test_recent_deployment(trace_id, input, output, metadata):
    """Prioritize evaluating the most recent traces."""
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("Recent-First Sampling (decay=0.8):\n")
test_recent_deployment()

---

## 6. No Sampling (Evaluate Everything)

**Best for:** Small projects, post-incident audits, or when you need 100% coverage.

This is the default if you don't specify a `sampling` parameter.

> **Colab Note:** Always combine with `limit` or a narrow `time_range` to avoid evaluating thousands of traces by accident.

In [ ]:
# Explicit NoSamplingStrategy (same as not specifying sampling at all)
@fe.correctness(threshold=0.8)
@langfuse_traces(
    time_range="last_1h",
    sampling=NoSamplingStrategy(),  # Evaluate ALL traces in the window
    limit=10,                       # Safety net: max 10 traces
)
def test_full_audit(trace_id, input, output, metadata):
    """Evaluate every trace from the last hour."""
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("No Sampling (all traces in last 1h, max 10):\n")
test_full_audit()

---

## 7. Comparison Table

| Strategy | Best For | Cost Impact | Bias |
|----------|----------|-------------|------|
| **Random** | General monitoring | Low (fixed sample size) | None (unbiased) |
| **Stratified** | Cross-segment fairness | Medium (samples per group) | None within groups |
| **Score-Based** | Failure investigation | Very low (focus on failures) | Intentional (oversamples failures) |
| **Recent-First** | Post-deploy monitoring | Low-Medium | Recency bias |
| **No Sampling** | Full audits, small projects | High (all traces) | None |

---

## 8. Scheduling Patterns

Different strategies fit different schedules. Here are three common patterns:

### Pattern A: Hourly Quick Check

- **Goal:** Is the system healthy *right now*?
- **Strategy:** `RandomSamplingStrategy(sample_size=20)`
- **Metrics:** 1-2 core metrics (`correctness`)
- **Time range:** `last_1h`
- **Cost:** ~20 judge calls/hour = ~480/day

In [ ]:
# Pattern A: Hourly Quick Check
@fe.correctness(threshold=0.8)
@langfuse_traces(
    time_range="last_1h",
    sampling=RandomSamplingStrategy(sample_size=20),
)
def test_hourly_health_check(trace_id, input, output, metadata):
    """Quick hourly check: is the system healthy?"""
    fe.score(output, input=input)


print("Hourly Quick Check pattern:")
print("  Strategy:   Random, 20 traces")
print("  Metrics:    correctness only")
print("  Time range: last_1h")
print("  Est. cost:  ~20 judge calls")

### Pattern B: Nightly Deep Dive

- **Goal:** Comprehensive quality report
- **Strategy:** `ScoreBasedSamplingStrategy` or `StratifiedSamplingStrategy`
- **Metrics:** Full stack (3-5 metrics)
- **Time range:** `last_24h`
- **Cost:** ~500-1000 judge calls/night

In [ ]:
# Pattern B: Nightly Deep Dive
@fe.correctness(threshold=0.8)
@fe.relevance(threshold=0.8)
@fe.hallucination(threshold=0.9)
@langfuse_traces(
    time_range="last_24h",
    sampling=ScoreBasedSamplingStrategy(
        score_name="user_rating",
        low_score_threshold=3.0,
        low_score_rate=1.0,
        high_score_rate=0.1,
    ),
)
def test_nightly_deep_dive(trace_id, input, output, context, metadata):
    """Nightly deep dive: full metric stack, focus on failures."""
    fe.score(output, context=context, input=input)


print("Nightly Deep Dive pattern:")
print("  Strategy:   ScoreBased (100% failures, 10% successes)")
print("  Metrics:    correctness + relevance + hallucination")
print("  Time range: last_24h")
print("  Est. cost:  ~500-1000 judge calls")

### Pattern C: Post-Incident Audit

- **Goal:** What happened during the incident?
- **Strategy:** `NoSamplingStrategy` (evaluate everything)
- **Metrics:** Full stack
- **Time range:** Specific absolute window
- **Cost:** Depends on incident duration and traffic

In [ ]:
# Pattern C: Post-Incident Audit
@fe.correctness(threshold=0.7)
@fe.hallucination(threshold=0.9)
@langfuse_traces(
    time_range="2026-03-04 14:00 to 2026-03-04 16:00",
    sampling=NoSamplingStrategy(),
)
def test_incident_audit(trace_id, input, output, context, metadata):
    """Post-incident: evaluate every trace in the window."""
    fe.score(output, context=context, input=input)


print("Post-Incident Audit pattern:")
print("  Strategy:   NoSampling (evaluate everything)")
print("  Metrics:    correctness + hallucination")
print("  Time range: 2026-03-04 14:00 to 16:00")
print("  Est. cost:  depends on traces in window")

---

## 9. GitHub Actions Automation

Automate your evaluation schedule with GitHub Actions. Here's a production-ready workflow:

```yaml
# .github/workflows/nightly-eval.yml
name: Nightly Production Evaluation

on:
  schedule:
    - cron: '0 8 * * *'   # 8 AM UTC daily
  workflow_dispatch:        # Manual trigger for ad-hoc runs

jobs:
  evaluate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: pip install fasteval-core fasteval-langfuse pytest

      - name: Run nightly evaluation
        env:
          LANGFUSE_PUBLIC_KEY: ${{ secrets.LANGFUSE_PUBLIC_KEY }}
          LANGFUSE_SECRET_KEY: ${{ secrets.LANGFUSE_SECRET_KEY }}
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
        run: pytest tests/test_nightly_eval.py -v --tb=short
```

### Tips for automation

- **Always set `limit` or `sample_size`** — prevents surprise bills if traffic spikes overnight
- **Use `gpt-4o-mini`** for routine checks — much cheaper, often good enough for monitoring
- **Save `gpt-4o`** for nightly deep dives or post-incident audits where quality matters more
- **Run fewer metrics hourly** (1-2), **full stack nightly** (3-5)
- **Use `seed`** in Random sampling for reproducible CI runs

---

## 10. Cost Optimization Cheat Sheet

| Lever | Impact | Example |
|-------|--------|---------|
| **Reduce sample size** | Linear | 200 → 50 traces = 75% savings |
| **Fewer metrics** | Linear | 5 → 2 metrics = 60% savings |
| **Cheaper judge model** | ~10x | GPT-4o → GPT-4o-mini |
| **Score-based sampling** | ~20x | Evaluate 100% failures + 5% successes |
| **Hourly + nightly split** | ~5x | Lightweight hourly, deep nightly |

**Rule of thumb:** Start with `RandomSamplingStrategy(sample_size=50)` and 1-2 metrics. Add more as you understand your quality patterns.

---

## Next Steps

| Notebook | What you'll learn |
|----------|-------------------|
| [Basic Trace Evaluation](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/basic-trace-evaluation.ipynb) | Metric stacking, handling missing data, score reporting |
| [Dataset Regression Testing](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/dataset-regression-testing.ipynb) | Golden datasets, A/B testing, Langfuse Experiments, CI/CD gates |

For the full API reference, see the [fasteval-langfuse plugin docs](https://fasteval.dev/docs/plugins/langfuse/overview).